In [2]:
!nvidia-smi

Fri Apr 24 11:23:01 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   77C    P0              36W /  70W |  14922MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
!pip install transformers
!pip install 'accelerate>=1.1.0'

In [11]:
import os
print("현재 경로:", os.getcwd())
print("현재 폴더 내용:", os.listdir())

현재 경로: /home/jovyan/work/DLThon_DKTC_Team4/Dohyun
현재 폴더 내용: ['KcELECTRA.ipynb', '.ipynb_checkpoints', 'DATA preprocessing.ipynb', 'results', 'KLUE_RoBERTa.ipynb', 'DATA_EDA.ipynb', 'README.md', 'data']


In [15]:
import pandas as pd
train_combined_path = '../Xuxeong/final_data.csv'
df = pd.read_csv(train_combined_path)

# 1. 컬럼 이름 확인하기
print("데이터 컬럼명:", df.columns.tolist())

# 2. 데이터 앞부분 살짝 보기
print(df.head())

print(len(df))

데이터 컬럼명: ['conversation', 'label']
                                        conversation  label
0  야!.야!야야!! 네? 너 몇학년이야? 1학년이요. 1학년? 선배가 부르는데 대답을...      1
1  방에서 냄새나는 거 같아서 양키캔들 하나 샀어. 오 무슨 향 샀는데? 라벤더나 코튼...      4
2  김대리 이번 주말에 뭐하나? 주말에 여자친구랑 데이트가기로했습니다 그약속다음주로 미...      2
3  아 겨울 패딩 꺼냈는데 지퍼가 고장 나서 안 올라가. 지퍼 이빨이 나간 거야, 아니...      4
4  강대리 나 바빠서 그러는데 이것좀 부탁해 네 알겠습니다 거기 이름은 내이름으로 적고...      2
5035


In [17]:
# 라벨별 개수 (빈도수)
print("--- 라벨별 데이터 개수 ---")
print(df['label'].value_counts())

# 라벨별 비율 (Percentage)
print("\n--- 라벨별 데이터 비율 ---")
print(df['label'].value_counts(normalize=True) * 100)

--- 라벨별 데이터 개수 ---
label
4    1190
3    1010
1     973
2     970
0     892
Name: count, dtype: int64

--- 라벨별 데이터 비율 ---
label
4    23.634558
3    20.059583
1    19.324727
2    19.265144
0    17.715988
Name: proportion, dtype: float64


In [4]:
import pandas as pd
import torch
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# ---------------------------------------------
# 0. 메모리 초기화
# ---------------------------------------------
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------
train_combined_path = './data/train_combined.csv'
df = pd.read_csv(train_combined_path)
df = df.dropna(subset=['conversation', 'label'])

# Train / Validation 분리 (8:2)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# ---------------------------------------------
# 2. 모델 및 토크나이저 로드 (하이퍼파라미터 원본 유지)
# ---------------------------------------------
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 5          
LEARNING_RATE = 1e-5

MODEL_NAME = "beomi/KcELECTRA-base-v2022" 
num_labels = len(df['label'].unique()) 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# ---------------------------------------------
# 3. Custom Dataset 클래스
# ---------------------------------------------
class KcElectraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = KcElectraDataset(train_df['conversation'].values, train_df['label'].values, tokenizer, MAX_LENGTH)
val_dataset = KcElectraDataset(val_df['conversation'].values, val_df['label'].values, tokenizer, MAX_LENGTH)

# ---------------------------------------------
# 4. 평가지표 계산 함수
# ---------------------------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    
    return {
        'accuracy': acc,
        'f1_macro': f1
    }

# ---------------------------------------------
# 5. Trainer 셋팅 및 학습 (Training)
# ---------------------------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,                    # EPOCHS = 5
    per_device_train_batch_size=32,        # BATCH_SIZE = 32
    per_device_eval_batch_size=32,
    learning_rate=1e-5,                    # LEARNING_RATE = 1e-5
    weight_decay=0.01,                     # Weight Decay = 0.01
    adam_epsilon=1e-5,                     # Epsilon = 1e-5
    max_grad_norm=1.0,                     # Max Grad Norm = 1.0
    warmup_ratio=0.1,                      # SCHEDULER = 10% Warmup
    lr_scheduler_type="linear",            # SCHEDULER = Linear
    seed=42,                               # SEED = 42
    eval_strategy="epoch",                 # (에러 수정됨) 에포크마다 평가
    save_strategy="epoch",                 
    logging_steps=50,                      
    report_to="none"                       
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("▶ [1/2] 모델 학습을 시작합니다...\n")
trainer.train()

# ==========================================
# 6. 최종 평가 및 리포트 출력 (Evaluation)
# ==========================================
print("\n▶ [2/2] 최종 모델 성능 평가를 진행합니다...")

# 기본 metrics 확인 (eval_loss, eval_accuracy 등)
eval_results = trainer.evaluate()

print("-" * 50)
print("🎯 [최종 검증 세트(Validation Set) 평가 결과]")
print(f" - Loss (손실): {eval_results['eval_loss']:.4f}")
print(f" - Accuracy (정확도): {eval_results['eval_accuracy'] * 100:.2f}%")
print(f" - F1 Score (Macro): {eval_results['eval_f1_macro']:.4f}")
print("-" * 50)

print("\n▶ 검증 데이터셋 상세 예측 중...")
# val_dataset에 대해 예측을 수행하여 로짓(logits) 추출
output = trainer.predict(val_dataset)
preds = np.argmax(output.predictions, axis=-1)

# 타겟 이름 맵핑 (0~4번 인덱스 순서에 맞게 설정)
target_names = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']

print("\n" + "="*60)
print("🎯 [KcELECTRA 상세 성능 리포트]")
print("="*60)
# 실제 정답(val_df['label'])과 모델의 예측값(preds) 비교
print(classification_report(val_df['label'].values, preds, target_names=target_names, zero_division=0))
print("="*60)

# 학습된 모델 및 토크나이저 안전하게 저장
save_directory = "./best_kcelectra_model2"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"\n✅ 학습 완료! 모델과 토크나이저가 '{save_directory}' 폴더에 안전하게 저장되었습니다.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

▶ [1/2] 모델 학습을 시작합니다...



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.343188,0.987901,0.792929,0.794844
2,0.744134,0.449017,0.903030,0.903267
3,0.385180,0.325083,0.919192,0.918232
4,0.294313,0.297191,0.925253,0.924204
5,0.245682,0.285237,0.922222,0.921763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.245682,0.285237,5,0.922222,0.921763


--------------------------------------------------
🎯 [최종 검증 세트(Validation Set) 평가 결과]
 - Loss (손실): 0.2852
 - Accuracy (정확도): 92.22%
 - F1 Score (Macro): 0.9218
--------------------------------------------------

▶ 검증 데이터셋 상세 예측 중...



🎯 [KcELECTRA 상세 성능 리포트]
              precision    recall  f1-score   support

       협박 대화       0.87      0.88      0.87       179
       갈취 대화       0.87      0.89      0.88       196
 직장 내 괴롭힘 대화       0.96      0.94      0.95       196
   기타 괴롭힘 대화       0.90      0.90      0.90       219
       일반 대화       1.00      1.00      1.00       200

    accuracy                           0.92       990
   macro avg       0.92      0.92      0.92       990
weighted avg       0.92      0.92      0.92       990



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ 학습 완료! 모델과 토크나이저가 './best_kcelectra_model2' 폴더에 안전하게 저장되었습니다.


#### Submission

In [ ]:
import pandas as pd

test_df = pd.read_csv("data/test.csv")
submission_df = pd.read_csv("data/submission_default.csv")

print(f"📄 문제지(test.csv) 행 개수: {len(test_df)}개")
print(f"📝 답안지(submission.csv) 행 개수: {len(submission_df)}개")

In [5]:
import pandas as pd
from transformers import pipeline

# 1. 파일 로드 및 모델 준비
test_df = pd.read_csv("data/test.csv")
submission_df = pd.read_csv("data/submission_default.csv")

classifier = pipeline(
    "text-classification", 
    model=trainer.model, 
    tokenizer=tokenizer, 
    device=0 
)

# 2. 추론 실행 
print(f"총 {len(test_df)}건 추론 중...")
results = classifier(test_df['conversation'].tolist(), batch_size=16)

# 3. 결과 정리 (핵심 변경 부분!)
# res['label']은 'LABEL_0' 형태로 나오므로, 문자열에서 숫자만 추출(int)합니다.
# 예: 'LABEL_0' -> 0
predictions = [int(res['label'].split('_')[-1]) for res in results]

# 4. submission 양식에 채우기
submission_df['class'] = predictions

# 5. 저장 (숫자만 들어가므로 인코딩 옵션은 빼도 무방합니다)
output_path = "data/submission_KcELECTRA2.csv"
submission_df.to_csv(output_path, index=False)

print("-" * 50)
print(f"✅ 제출 파일 생성 완료: {output_path}")

# 6. 결과 확인
print(submission_df.head(10))

총 500건 추론 중...
--------------------------------------------------
✅ 제출 파일 생성 완료: data/submission_KcELECTRA2.csv
     idx  class
0  t_000      1
1  t_001      2
2  t_002      2
3  t_003      3
4  t_004      3
5  t_005      0
6  t_006      0
7  t_007      1
8  t_008      3
9  t_009      1


#### 오답노트

In [11]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import classification_report

print("검증 데이터 기반 최종 상세 평가 및 확신도 분석 중...")

# 1. 평가 데이터에 대해 예측 수행 (Logit 값 추출)
predictions = trainer.predict(val_dataset)
logits = predictions.predictions
true_labels = predictions.label_ids

# 2. Logit을 Softmax로 변환하여 확률(확신도) 계산
probs = softmax(logits, axis=1)
pred_labels = np.argmax(probs, axis=1)         # 가장 높은 확률의 라벨(0~4)
confidences = np.max(probs, axis=1) * 100      # 가장 높은 확률의 퍼센트(%)

# 3. 기본 성적표 출력
target_names = ['협박(0)', '갈취(1)', '직장내괴롭힘(2)', '기타괴롭힘(3)', '일반대화(4)']
print("\n" + "="*50)
print("[클래스별 상세 성적표]")
print("="*50)
print(classification_report(true_labels, pred_labels, target_names=target_names))

# =====================================================================
# 모델의 확신도를 바탕으로 한 '오답 노트' 생성
# =====================================================================

# 원본 텍스트를 가져와서 데이터프레임으로 묶기 (val_df 사용)
error_analysis_df = pd.DataFrame({
    '문장': val_df['conversation'].values,
    '정답': [target_names[i] for i in true_labels],
    '예측': [target_names[i] for i in pred_labels],
    '확신도(%)': np.round(confidences, 2)
})

# 4-1. 모델이 틀린 문제만 필터링
wrong_preds = error_analysis_df[error_analysis_df['정답'] != error_analysis_df['예측']]

# 4-2. 확신도가 높은 순서대로 정렬 (완전 확신했는데 틀린 '최악의 오답')
worst_mistakes = wrong_preds.sort_values(by='확신도(%)', ascending=False)

print("\n" + "="*80)
print("[핵심 오답 노트] 모델이 '완벽하게 확신했는데' 틀린 Top 5 문장")
print("   (이 문장들을 보면 모델이 어떤 패턴을 오해하고 있는지 알 수 있습니다!)")
print("="*80)

# 문장이 길어도 잘리지 않고 다 보이게 설정
pd.set_option('display.max_colwidth', None)
print(worst_mistakes.head(5).to_string(index=False))
pd.reset_option('display.max_colwidth')

# 전체 오답 노트를 CSV 파일로 저장
worst_mistakes.to_csv("model_worst_mistakes.csv", index=False, encoding='utf-8-sig')
print("\n오답 노트가 'model_worst_mistakes.csv'로 저장되었습니다.")

검증 데이터 기반 최종 상세 평가 및 확신도 분석 중...



[클래스별 상세 성적표]
              precision    recall  f1-score   support

       협박(0)       0.86      0.90      0.88       178
       갈취(1)       0.89      0.88      0.88       195
   직장내괴롭힘(2)       0.94      0.93      0.94       194
    기타괴롭힘(3)       0.87      0.87      0.87       202
     일반대화(4)       0.99      0.98      0.99       179

    accuracy                           0.91       948
   macro avg       0.91      0.91      0.91       948
weighted avg       0.91      0.91      0.91       948


[핵심 오답 노트] 모델이 '완벽하게 확신했는데' 틀린 Top 5 문장
   (이 문장들을 보면 모델이 어떤 패턴을 오해하고 있는지 알 수 있습니다!)
                                                                                                                                                                                                                                                                                                                                        문장       정답       예측    확신도(%)
                         아 얘봐 얘 신발봐 하하하하 진짜 신발 대박 꼴